# 02 - Data Preparation

Cleaning checks and the dataset-size decision (sample vs. keep the full 800,000 rows). No train/validation/test split is saved to disk: each modelling notebook (`04`, `05`, `06`) re-derives the identical split inline from `dataset_pe_v2_full.csv`, using the same fixed `RANDOM_STATE`, rather than persisting extra split CSVs.


In [8]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv("../data/dataset_pe_v2_full.csv")
feature_cols = [c for c in df.columns if c not in ("Name", "Malware", "OriginalSplit")]
print("shape:", df.shape)

shape: (800000, 23)


## Cleaning checks

Standard pandas checks first (`df.isna()`, `df.duplicated()`), then two checks specific to this dataset's structure: duplicate SHA-256 file hashes (a stronger notion of "the same file" than an exact-row duplicate would catch) and zero-variance / infinite feature values (not caught by a generic `isna()` check).

In [9]:
# Missing values
na_counts = df.isna().sum()
print("df.isna().sum() -> total missing values:", int(na_counts.sum()))
print(na_counts[na_counts > 0] if na_counts.sum() > 0 else "no missing values in any column")


df.isna().sum() -> total missing values: 0
no missing values in any column


In [10]:
# Duplicate rows
full_row_dupes = df.duplicated().sum()
print("df.duplicated().sum() -> fully duplicated rows:", int(full_row_dupes))

name_dupes = df["Name"].duplicated().sum()
print("df['Name'].duplicated().sum() -> duplicate SHA-256 hashes:", int(name_dupes))
print("(checked separately from the full-row check above: two rows could differ on a "
      "derived column yet still represent the same underlying file, so the hash check "
      "is the stronger, dataset-specific test)")


df.duplicated().sum() -> fully duplicated rows: 0
df['Name'].duplicated().sum() -> duplicate SHA-256 hashes: 0
(checked separately from the full-row check above: two rows could differ on a derived column yet still represent the same underlying file, so the hash check is the stronger, dataset-specific test)


In [11]:
# Apply the standard cleaning actions. Both are no-ops today given the checks above,
# but are run for real (not skipped) so the notebook performs actual cleaning rather
# than only reporting diagnostics, and so it stays correct if a future data refresh
# does introduce either issue.
before = len(df)
df = df.dropna()
df = df.drop_duplicates(subset="Name")
after = len(df)
print(f"rows before cleaning: {before}")
print(f"rows after df.dropna() + df.drop_duplicates(subset='Name'): {after}")
print(f"rows dropped: {before - after}")


rows before cleaning: 800000
rows after df.dropna() + df.drop_duplicates(subset='Name'): 800000
rows dropped: 0


In [12]:
# Zero-variance feature columns (no signal, only noise; not caught by isna/duplicated)
zero_var = [c for c in feature_cols if df[c].std() == 0]
print("zero-variance feature columns:", zero_var if zero_var else "none")


zero-variance feature columns: none


In [13]:
# Infinite values (not caught by isna()/isnull(), checked separately)
inf_count = np.isinf(df[feature_cols]).sum().sum()
print("infinite feature values:", int(inf_count))


infinite feature values: 0


No rows need to be dropped: 0 missing values (`df.isna()`), 0 full-row duplicates (`df.duplicated()`), 0 duplicate SHA-256 hashes, 0 zero-variance columns, 0 infinite values. `df.dropna()` and `df.drop_duplicates()` were run for real and confirmed to be no-ops. `dataset_pe_v2_full.csv` is used as-is by every downstream notebook, no separate cleaned copy is needed.


## Dataset size decision: keep the full 800,000 rows, no downsampling

**Reasoning, based on published practice:**

1. **The EMBER paper's own baseline (Anderson & Roth, 2018, arXiv:1804.04637) trains on the complete labeled training corpus, not a subsample**, and reports ROC AUC 0.999 with well under 1% false positives at over 98% detection. Downsampling was never part of how that benchmark number was produced, so shrinking this project's dataset for convenience would be an avoidable step backward from the reference this project is benchmarked against.
2. **Tree ensembles (Random Forest, XGBoost) scale close to linearly with row count** and routinely train on hundreds of thousands to millions of rows within minutes on ordinary laptop hardware, a well-established property of these algorithms. 800,000 rows by 20 features is not a genuine compute obstacle for this model family.
3. **More real, labeled data directly reduces the risk of a benign class that does not structurally resemble typical real-world software**, one of the central goals of this project. Shrinking the dataset only makes sense when there is an actual compute or time constraint forcing the trade-off; there is not one here.

**Where a smaller subsample is still legitimately used later:** `06_evaluation.ipynb`'s SHAP explainability step uses a random subsample (a few thousand rows) purely because SHAP summary/beeswarm plots do not get more informative or readable past a certain point and full-scale plotting is wasted compute, not because the training data itself is being reduced. That is a plotting-practicality decision, not a data-quantity decision, and it happens after training, not before. Separately, `04`/`05`/`06`'s `GridSearchCV`/training step downsamples the ~600,000-row training pool to 150,000 rows (75,000 per class) purely for tuning-time compute reasons on ordinary laptop hardware; this is a disclosed, documented trade-off (see the note directly above the split cell below), and it does not touch the held-out EMBER `test` split, so final reported test metrics in `06` remain on the full, standard 200,000-row benchmark split.


## Train / validation / test split (reference only, not persisted)

`04`, `05`, and `06` each independently reload `dataset_pe_v2_full.csv` and derive the same split shown below, using this fixed `RANDOM_STATE` (no split files saved to disk, every notebook regenerates the exact same rows on its own). EMBER's own official `train`/`test` boundary (`OriginalSplit` column) is kept as the outer split (matches the published benchmark, avoids introducing leakage via an in-house re-split); an 85/15 stratified split of the `train` rows produces the validation set. Shown here once for reference.


**Compute-constrained downsample of the training pool.** `GridSearchCV`'s exhaustive search over the full ~600,000-row training pool is too slow on typical laptop hardware (each grid search trains many separate models across cross-validation folds). Downsamples `train_pool` to 75,000 rows per class (150,000 total) before the 85/15 train/validation split, using a fixed `RANDOM_STATE` so the reduction is reproducible. **The held-out `test` set (EMBER's own official 200,000-row split) is NOT reduced**: `06_evaluation.ipynb`'s final reported numbers still come from the complete, standard benchmark split, only the tuning step works on a smaller pool.


In [14]:
train_pool = df[df["OriginalSplit"] == "train"].reset_index(drop=True)
test_df = df[df["OriginalSplit"] == "test"].reset_index(drop=True)

TRAIN_POOL_SAMPLE_PER_CLASS = 75000
train_pool = pd.concat([
    g.sample(n=min(TRAIN_POOL_SAMPLE_PER_CLASS, len(g)), random_state=RANDOM_STATE)
    for _, g in train_pool.groupby("Malware")
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("downsampled train_pool:", train_pool.shape)

train_df = pd.concat([
    g.sample(frac=0.85, random_state=RANDOM_STATE)
    for _, g in train_pool.groupby("Malware")
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
val_df = train_pool.drop(
    pd.concat([g.sample(frac=0.85, random_state=RANDOM_STATE) for _, g in train_pool.groupby("Malware")]).index
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pct = d["Malware"].value_counts(normalize=True).sort_index() * 100
    print(f"{name}: n={len(d)}  benign={pct[0]:.2f}%  malicious={pct[1]:.2f}%")

downsampled train_pool: (150000, 23)
train: n=127500  benign=50.00%  malicious=50.00%
val: n=22500  benign=50.00%  malicious=50.00%
test: n=200000  benign=50.00%  malicious=50.00%


## Summary

No rows dropped (0 duplicates, 0 zero-variance, 0 invalid values), full 800,000-row dataset retained (decision justified above). Reference split (training pool downsampled to 150,000 rows for tuning compute, see note above): ~127,500 train / ~22,500 validation / 200,000 held-out test, stratified by class, test set is EMBER's own official split, untouched. Not saved to disk, `04`/`05`/`06` each reproduce it inline with the same `RANDOM_STATE`. Next: `03_eda.ipynb` explores feature distributions and correlations.
